In [ ]:
from typing import Callable

class TaskPython:
    def __init__(self, name:str, fn:Callable):
        self.name = name
        self.successors = {}
        self.next_action = 'default'
        self.fn = fn

    def __call__(self):
        return self.fn()

    # def __sub__(self, next_action_name:str):
    #     self.next_action = next_action_name
    #     return self

    def __rshift__(self, other):
        self.successors[self.next_action] = other
        return other

def TaskOperator(name:str, fn:Callable):
    return TaskPython(name=name, fn=fn)

In [2]:
task1 = TaskOperator(name="Task 1", fn=lambda: print("Executing Task 1"))
task2 = TaskOperator(name="Task 2", fn=lambda: print("Executing Task 2"))
task3 = TaskOperator(name="Task 3", fn=lambda: print("Executing Task 3"))

In [3]:
task1, task2, task3

(<__main__.TaskPython at 0x197b279a990>,
 <__main__.TaskPython at 0x197b279ae10>)

In [4]:
task1 >> task2 >> task3

In [5]:
task1(), task2(), task3()

Executing Task 1
Executing Task 2
Executing Task 3


(None, None, None)

In [6]:
for t in [task1, task2, task3]:
    print(f"Task: {t.name}, Successors: {list(t.successors.keys())}")

Task: Task 1, Successors: ['default']
Task: Task 2, Successors: ['default']
Task: Task 3, Successors: []


In [7]:
task2.successors

{'default': <__main__.TaskPython at 0x197b279ae10>}

In [13]:
import subprocess

class TaskBash:
    def __init__(self, name:str, command:str):
        self.name = name
        self.successors = {}
        self.next_action = 'default'
        self.command = command

    def __call__(self):
        result = subprocess.run(self.command, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"[{self.name}] exited {result.returncode}: {result.stderr}")
        return result.stdout

    def __rshift__(self, other):
        self.successors[self.next_action] = other
        return other

def BashOperator(name:str, command:str):
    return TaskBash(name=name, command=command)

In [18]:
task1 = BashOperator(name="Task 1", command="uv run ../examples/scripts/task1.py")
task2 = BashOperator(name="Task 2", command="uv run ../examples/scripts/task2.py")
task3 = BashOperator(name="Task 3", command="uv run ../examples/scripts/task3.py")

In [20]:
task1, task2, task3

(<__main__.TaskBash at 0x197b27d66c0>,
 <__main__.TaskBash at 0x197b27d6030>)

In [21]:
task1 >> task2 >> task3

In [19]:
task1(), task2(), task3()  # This will execute the tasks in order

('Task 1 executed\n', 'Task 2 executed\n', 'Task 3 executed\n')

In [22]:
for t in [task1, task2, task3]:
    print(f"Task: {t.name}, Successors: {list(t.successors.keys())}")

Task: Task 1, Successors: ['default']
Task: Task 2, Successors: ['default']
Task: Task 3, Successors: []


In [26]:
next_run = True
task_run = task1
while next_run:
    print(task_run())
    if task_run.successors:
        task_run = task_run.successors[task_run.next_action]
    else:
        next_run = False

Task 1 executed

Task 2 executed

Task 3 executed



In [28]:
from typing import Callable

class TaskPython:
    def __init__(self, name:str, fn:Callable):
        self.name = name
        self.successors = {}
        self.next_action = 'default'
        self.fn = fn

    def __call__(self):
        return self.fn()

    def __sub__(self, next_action_name:str):
        self.next_action = next_action_name
        return self

    def __rshift__(self, other):
        self.successors[self.next_action] = other
        return other

def TaskOperator(name:str, fn:Callable):
    return TaskPython(name=name, fn=fn)

In [48]:
task1 = TaskOperator(name="Task 1", fn=lambda: print("Executing Task 1"))
task2 = TaskOperator(name="Task 2", fn=lambda: print("Executing Task 2"))
task3 = TaskOperator(name="Task 3", fn=lambda: print("Executing Task 3"))
task4 = TaskOperator(name="Task 4", fn=lambda: print("Executing Task 4"))

In [ ]:
_ = task1 >> task2 >> task4
_ = task1 - "cond1" >> task3 >> task4
_ = task1 - "cond2" >> task4

In [47]:
task1.successors, task2.successors, task3.successors

({'default': <__main__.TaskPython at 0x197b28f43b0>,
  'cond1': <__main__.TaskPython at 0x197b2a08800>},
 {'default': <__main__.TaskPython at 0x197b2a08800>},
 {})